# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
from importlib.metadata import PackageNotFoundError, version

required_packages = {
    "transformers": "4.57.1",
    "sentencepiece": "0.2.1",
    "safetensors": "0.6.2",
    "tokenizers": "0.22.1",
    "huggingface-hub": "0.36.0",
}
install_packages = []
for package, required_version in required_packages.items():
    try:
        installed_version = version(package)
    except PackageNotFoundError:
        installed_version = None
    if installed_version != required_version:
        install_packages.append(f"{package}=={required_version}")
try:
    version("torch")
except PackageNotFoundError:
    install_packages.append("torch==2.9.1")
if install_packages:
    get_ipython().run_line_magic("pip", "install -q " + " ".join(install_packages))
else:
    print("Required packages are already installed.")

Required packages are already installed.


In [2]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import torch
from transformers import (
    AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification, AutoTokenizer, pipeline, set_seed,
)
from transformers.utils import logging

logging.set_verbosity_error()
torch.set_num_threads(min(4, os.cpu_count() or 1))
print("Libraries loaded successfully!")

D:\IAU\Graduation_Project\tmp\nlp-lab-work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
decoder_model_id = "distilbert/distilgpt2"
decoder_tokenizer = AutoTokenizer.from_pretrained(decoder_model_id)
decoder_model = AutoModelForCausalLM.from_pretrained(
    decoder_model_id, use_safetensors=True
)
decoder_model.eval()
generator = pipeline(
    "text-generation", model=decoder_model,
    tokenizer=decoder_tokenizer, device=-1,
)

prompt = "Natural language processing is"
output = generator(
    prompt, max_new_tokens=30, do_sample=False,
    pad_token_id=decoder_tokenizer.eos_token_id,
)
print(output[0]["generated_text"])

Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small", use_safetensors=True
)
model.eval()

translation_prompt = "Translate to French: I love natural language processing."
translation_inputs = tokenizer(translation_prompt, return_tensors="pt")
with torch.inference_mode():
    translation_output = model.generate(
        **translation_inputs, max_new_tokens=60, do_sample=False
    )
translation = tokenizer.decode(translation_output[0], skip_special_tokens=True)
print(translation)

Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
encoder_model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
encoder_tokenizer = AutoTokenizer.from_pretrained(encoder_model_id)
encoder_model = AutoModelForSequenceClassification.from_pretrained(
    encoder_model_id, use_safetensors=True
)
encoder_model.eval()
classifier = pipeline(
    "sentiment-analysis", model=encoder_model,
    tokenizer=encoder_tokenizer, device=-1,
)

text = "This NLP lab is easy and interesting."
result = classifier(text)
print(result)

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [6]:
model_families = {
    "Sentiment classification": "Encoder-only",
    "Machine translation": "Encoder-decoder",
    "Open-ended text generation": "Decoder-only",
    "Named Entity Recognition": "Encoder-only",
    "Text summarization": "Encoder-decoder",
}
for number, (task, family) in enumerate(model_families.items(), start=1):
    print(f"{number}. {task}: {family}")

1. Sentiment classification: Encoder-only
2. Machine translation: Encoder-decoder
3. Open-ended text generation: Decoder-only
4. Named Entity Recognition: Encoder-only
5. Text summarization: Encoder-decoder


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [7]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]
sentiment_results = classifier(sentences)
for sentence, prediction in zip(sentences, sentiment_results):
    print(sentence)
    print(f"Label: {prediction['label']} | Score: {prediction['score']:.6f}")
    print()

The movie was excellent.
Label: POSITIVE | Score: 0.999861

The lecture was difficult to understand.
Label: NEGATIVE | Score: 0.999665

I enjoyed learning about transformers.
Label: POSITIVE | Score: 0.999746



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [8]:
paragraph = (
    "A university library introduced an online booking system for study rooms. "
    "Students can now reserve rooms from their phones and check availability "
    "before visiting the library. The system reduces waiting times and helps "
    "students organize group study sessions more easily."
)
summary_prompt = "Write a short summary of this text: " + paragraph
summary_inputs = tokenizer(
    summary_prompt, return_tensors="pt", truncation=True, max_length=512
)
with torch.inference_mode():
    summary_output = model.generate(
        **summary_inputs, max_new_tokens=60, do_sample=False,
        num_beams=4, no_repeat_ngram_size=3,
    )
summary = tokenizer.decode(summary_output[0], skip_special_tokens=True)
print("Paragraph:", paragraph)
print("Summary:", summary)

Paragraph: A university library introduced an online booking system for study rooms. Students can now reserve rooms from their phones and check availability before visiting the library. The system reduces waiting times and helps students organize group study sessions more easily.
Summary: University library introduces online booking system for study rooms



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


A decoder-only model generates a response one token at a time using the conversation as context. Causal self-attention lets it use earlier tokens while predicting the next token. This makes it suitable for producing flexible, open-ended replies and continuing a dialogue. Instruction tuning and alignment training help it follow user requests and produce more useful responses.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [9]:
sampling_prompt = "Natural language processing is"
sampled_texts = []
for run in range(1, 4):
    sampled_output = generator(
        sampling_prompt, max_new_tokens=40, do_sample=True,
        temperature=0.8, top_p=0.9, top_k=50,
        repetition_penalty=1.1, no_repeat_ngram_size=3,
        pad_token_id=decoder_tokenizer.eos_token_id,
    )
    sampled_text = sampled_output[0]["generated_text"]
    sampled_texts.append(sampled_text)
    print(f"Run {run}: {sampled_text}")
    print()

Run 1: Natural language processing is a process in which the interpreter interprets an object, and then translates it into different languages.
The key to this approach is that both operators (for example) and operands (for instance),



Run 2: Natural language processing is the most common method for accessing a database of data in an Excel file. In this way, SQL Server does not only store all types of information such as user's Social Security numbers and current social security



Run 3: Natural language processing is the most important and fastest way to make a decision about your future career, as well as what you want done.



With sampling enabled, the model randomly selects each next token from its probability distribution instead of always choosing the most likely token. Without resetting a fixed random seed, successive runs can select different tokens even with the same prompt. Once a different token is selected, it changes the context for later predictions, so the final outputs can differ.